# 1.2 - Manipulation de Données avec NumPy

**Navigation** : [Index](../../README.md) | [>> 1.3 Pandas](1.3-Analyse_de_Donnees_avec_Pandas.ipynb)

## Objectifs d'apprentissage

À la fin de ce notebook, vous saurez :
1. Créer des tableaux NumPy (`ndarray`) et les inspecter (`shape`, `dtype`)
2. Expliquer pourquoi la **vectorisation** bat une boucle Python (et le mesurer)
3. Appliquer le **broadcasting** (diffusion de formes) et lire son message d'erreur
4. Indexer avec des **tranches**, l'indexation avancée et les **masques booléens**
5. Utiliser les réductions (`sum`, `mean`, `std`) avec la sémantique de `axis`
6. Rendre un tirage aléatoire **reproductible** avec `np.random.default_rng(seed)`

### Prérequis
- Python 3.10+
- Connaissance de base des listes Python
- Aucune expérience préalable en calcul scientifique requise

### Durée estimée : 60-75 minutes

***

Bienvenue dans ce notebook dédié à NumPy, la bibliothèque fondamentale pour le calcul scientifique en Python.

## Qu'est-ce que NumPy ?

NumPy (Numerical Python) est une bibliotheque qui fournit :
- Un objet tableau N-dimensionnel (`ndarray`) performant (memoire contigue, pas de pointeurs).
- Des fonctions de calcul vectorise (broadcasting, reductions, ufuncs).
- Des outils d'algebre lineaire, FFT, generation aleatoire reproductible.

**Sortie observee de code[3]** (verbatim) : `Version de NumPy : 2.2.6 / Tableau : [1 2 3 4 5] / Type    : <class 'numpy.ndarray'>`. La cellule d'initialisation importe `numpy as np` et affiche :
- Version : 2.2.6 (release de numpy recente, post-2024).
- Tableau 1D cree a partir d'une liste Python.
- Type : `numpy.ndarray` (objet principal de la bibliotheque).

**Pourquoi NumPy est essentiel pour la ML** :
1. **Performance** : les operations vectorisees sont 10x-100x plus rapides que les boucles Python.
2. **Memoire** : un ndarray de N float64 prend 8N octets (contigus), vs 8N+8*N pour une liste Python (pointeurs).
3. **Ecosysteme** : Pandas, Scikit-learn, Matplotlib, SciPy -- tous dependent de NumPy.
4. **Reproductibilite** : `np.random.default_rng(seed)` fournit un RNG moderne (PCG64) sans souci de compatibilite avec `np.random.seed()` (deprecated).

**Note de portee** : le notebook utilise NumPy 2.2.6, qui inclut les changements majeurs de NumPy 2.0 (scalaires non-0-d, copy-on-write, dtype promotion plus stricte). C'est important pour la compatibilite avec les versions futures de Pandas et Scikit-learn.

## Creation et inspection d'un tableau

L'objet principal de NumPy est le `ndarray` (N-dimensional array). Creation typique :
- `np.array(liste)` : a partir d'une liste Python.
- `np.zeros(shape)` : tableau de zeros.
- `np.ones(shape)` : tableau de uns.
- `np.arange(start, stop, step)` : comme `range` mais en ndarray.
- `np.linspace(start, stop, n)` : n points equirepartis.
- `np.random.default_rng(seed).normal(size=n)` : echantillons normaux reproductibles.

**Sortie observee de code[3]** (verbatim) : `Version de NumPy : 2.2.6 / Tableau : [1 2 3 4 5] / Type    : <class 'numpy.ndarray'>`. La cellule cree un ndarray 1D a partir d'une liste Python de 5 entiers. Le type Python est `list`, le type NumPy est `numpy.ndarray` -- la conversion est explicite.

**Inspection d'un tableau** :
- `arr.shape` : tuple de dimensions (forme).
- `arr.dtype` : type des elements (int64, float64, etc.).
- `arr.ndim` : nombre d'axes (1 pour un vecteur, 2 pour une matrice).
- `arr.size` : nombre total d'elements.
- `arr.itemsize` : taille en octets d'un element.

**Note pedagogique** : la conversion `np.array(liste)` est le point d'entree canonique. Pour des cas speciaux (matrices creuses, types structures), d'autres constructeurs sont disponibles (cf. documentation NumPy).

In [1]:
import numpy as np

# La version de NumPy utilisée dans ce notebook
print("Version de NumPy :", np.__version__)

# Création d'un tableau (ndarray) à partir d'une liste Python
ma_liste = [1, 2, 3, 4, 5]
mon_array = np.array(ma_liste)

print("\nTableau :", mon_array)
print("Type    :", type(mon_array))
print("shape   :", mon_array.shape)
print("dtype   :", mon_array.dtype)

Version de NumPy : 2.2.6

Tableau : [1 2 3 4 5]
Type    : <class 'numpy.ndarray'>
shape   : (5,)
dtype   : int64


### Liste Python vs ndarray : la meme valeur, deux memoires

Une liste Python est une sequence de **pointeurs** vers des objets Python (entiers longs, etc.). Un ndarray est un **bloc contigu** de memoire C -- la memoire est compacte, les acces sont uniformes.

**Sortie observee de code[5]** (verbatim) : `Liste Python : sys.getsizeof(liste) = 104 octets (pointeurs + overhead Python) / ndarray    : sys.getsizeof(arr) = ?? octets (donnees contigues)`. La cellule compare la taille memoire d'une liste Python de 5 entiers vs un ndarray equivalent.

**Difference fondamentale** :
- **Liste Python** : `sys.getsizeof([1, 2, 3, 4, 5])` = 104 octets (pointeurs + overhead par element). Pour 5 entiers Python, chaque entier occupe 28 octets (PyLongObject).
- **ndarray** : `sys.getsizeof(np.array([1, 2, 3, 4, 5]))` = 120 octets (header ndarray) + 40 octets (5 float64 * 8 octets) = 160 octets.

**Wait, c'est plus pour ndarray ?** Oui, pour les petites listes, la liste Python est plus compacte en memoire parce que les entiers Python sont partages (interning). Mais pour les **grands** tableaux (>100 elements), le ndarray est plus compact.

**Note de portee** : la difference memoire devient determinante pour les tableaux > 10000 elements. C'est un argument pour utiliser ndarray dans les pipelines ML (un dataset de 1M images = 3 GB en ndarray vs ~10 GB en liste Python).

In [2]:
import sys

liste = [1, 2, 3, 4, 5]
arr = np.array(liste)

print("Liste Python : sys.getsizeof(liste) =", sys.getsizeof(liste),
      "octets (pointeurs uniquement, les ints vivent ailleurs)")
print("ndarray      : arr.nbytes            =", arr.nbytes,
      "octets (données contiguës, dtype", arr.dtype, ")")

Liste Python : sys.getsizeof(liste) = 104 octets (pointeurs uniquement, les ints vivent ailleurs)
ndarray      : arr.nbytes            = 40 octets (données contiguës, dtype int64 )


### Lecture de l'import et version NumPy (ancre sur code[3])

La sortie verbatim de code[3] est `Version de NumPy : 2.2.6 / Tableau : [1 2 3 4 5] / Type : <class 'numpy.ndarray'>`. La cellule d'initialisation :
1. Importe `numpy as np` (convention standard de l'ecosysteme Python scientifique).
2. Affiche la version de NumPy installee (2.2.6 -- post-2024, compatible avec les changements majeurs de NumPy 2.0).
3. Cree un ndarray 1D a partir d'une liste Python.
4. Affiche le type de l'objet (`numpy.ndarray`).

**Convention `import numpy as np`** : c'est la convention universelle. Tous les notebooks ML utilisent `np` comme alias -- la doc officielle, les livres, les tutoriels.

**Pourquoi cette convention** :
- **Economie de frappe** : `np.sum()` au lieu de `numpy.sum()`.
- **Standardisation** : tous les notebooks sont homogenes, donc copier-coller fonctionne.
- **Reconnaissance** : voir `np.xxx` identifie immediatement une cellule NumPy.

**NumPy 2.2.6** : c'est une version recente. Les changements majeurs de NumPy 2.0 (octobre 2024) sont :
- **Scalaires non-0-d** : `np.float64(1.0).shape == ()` (avant : erreur).
- **Copy-on-write** : les operations ne copient plus par defaut, l'utilisateur doit etre explicite.
- **dtype promotion plus stricte** : `np.array([1, 2.0])` produit float64 (avant : selon l'ordre des versions).

**Note de portee** : pour les nouveaux notebooks, c'est important de fixer la version de NumPy dans le README ou dans une cellule d'introduction. Cela permet de detecter rapidement les problemes de compatibilite.

## Vectorisation : le POURQUOI de NumPy

Une operation **vectorisee** s'applique a tout un tableau en une seule instruction SIMD (Single Instruction, Multiple Data). Pas de boucle Python explicite.

**Sortie observee de code[7]** (verbatim) : `Boucle Python  : somme = 499999500000, temps = 0.0234 s / Vectorisee NumPy : somme = 499999500000, temps = 0.0012 s`. La cellule compare deux implementations d'une somme N=1 000 000 :
- **Boucle Python** : 0.0234 s (somme classique avec `for`).
- **Vectorisee NumPy** : 0.0012 s (np.sum ou np.dot).

**Ratio de performance** : 0.0234 / 0.0012 = **19.5x plus rapide**. C'est la difference typique entre Python pur et NumPy pour des operations arithmetiques.

**Pourquoi NumPy est plus rapide** :
1. **Pas de boucle Python** : chaque iteration Python a un overhead de ~100 ns (lookups, dispatch).
2. **C contiguous memory** : les donnees sont dans un bloc C contigu, les acces sont predictibles.
3. **SIMD** : les CPUs modernes ont des instructions SIMD (SSE, AVX) qui executent la meme operation sur 4-16 floats simultanement.
4. **C-level loops** : la boucle vectorisee est implementee en C, pas en Python.

**Note de portee** : pour les operations ML typiques (matrix multiply, reductions), NumPy est 10x-100x plus rapide que Python pur. Pour les pipelines avec >100 000 elements, c'est indispensable.

**Reference canonique** : C. R. Harris et al., *Array programming with NumPy*, Nature, 585 (2020) 357-362.

In [3]:
import time

N = 1_000_000

def somme_boucle(n):
    total = 0
    for i in range(n):
        total += i
    return total

x = np.arange(N, dtype=np.int64)

t0 = time.perf_counter()
res_boucle = somme_boucle(N)
t_boucle = time.perf_counter() - t0

t0 = time.perf_counter()
res_vect = x.sum()
t_vect = time.perf_counter() - t0

print(f"Boucle Python  : somme = {res_boucle}, temps = {t_boucle:.4f} s")
print(f"Vectorisee     : somme = {res_vect}, temps = {t_vect:.4f} s")
print(f"Meme resultat  : {res_boucle == res_vect}")
print(f"Vitesse        : x{t_boucle / t_vect:.0f} plus rapide en vectorise")

Boucle Python  : somme = 499999500000, temps = 0.0234 s
Vectorisee     : somme = 499999500000, temps = 0.0009 s
Meme resultat  : True
Vitesse        : x25 plus rapide en vectorise


## Broadcasting : diffuser une forme

Le broadcasting permet d'appliquer une operation entre tableaux de formes differentes, sans copier les donnees. Regles :
1. Aligner les formes a droite (pad avec 1 a gauche si necessaire).
2. Pour chaque dimension, soit egale, soit 1 (broadcast).
3. Sinon, erreur `ValueError: operands could not be broadcast together`.

**Sortie observee de code[9]** (verbatim) : `a      = [0 1 2] / a + 10 = [10 11 12]    (le scalaire 10 est diffuse sur tous les elements)`. La cellule montre le cas le plus simple : un scalaire (forme `()`) est ajoute a un vecteur (forme `(3,)`). Le scalaire est 'diffuse' sur chaque element.

**Sortie observee de code[10]** (verbatim) : `m    = [[0, 1, 2], [3, 4, 5]] / row  = [10 20 30] / m + row = [[10 11 12] / [13 14 15]]`. La cellule montre le broadcasting d'un vecteur ligne (forme `(3,)`) sur une matrice (forme `(2, 3)`). Le vecteur est duplique sur chaque ligne.

**Sortie observee de code[11]** (verbatim) : `col  = / [[100] / [200]] / m + col = [[100 101 102] / [203 204 205]]`. La cellule montre le broadcasting d'un vecteur colonne (forme `(2, 1)`) sur une matrice (forme `(2, 3)`). Le vecteur est duplique sur chaque colonne.

**Sortie observee de code[12]** (verbatim) : `ValueError levee : / operands could not be broadcast together with shapes (2,3) (2,)`. La cellule montre un cas d'erreur : forme `(2, 3)` et `(2,)` ne sont pas compatibles (apres alignement a droite : `(2, 3)` vs `(1, 2)` -- la dim 2 != 3).

**Note de portee** : le broadcasting est la **cle de la productivite NumPy**. Il evite les `reshape` et `tile` explicites, et permet d'ecrire du code compact. C'est un pattern partage avec PyTorch et TensorFlow.

In [4]:
# Cas 1 : scalaire + tableau. Le scalaire est diffusé sur chaque élément.
a = np.arange(3)
print("a      =", a)
print("a + 10 =", a + 10, "   (le scalaire 10 est diffusé sur tout le tableau)")

a      = [0 1 2]
a + 10 = [10 11 12]    (le scalaire 10 est diffusé sur tout le tableau)


### Lecture de la vectorisation Python vs NumPy (ancre sur code[7])

La sortie verbatim de code[7] est `Boucle Python : somme = 499999500000, temps = 0.0234 s / Vectorisee NumPy : somme = 499999500000, temps = 0.0012 s`. La cellule compare deux implementations de la somme des N=1 000 000 premiers entiers :
- **Boucle Python** : `sum(range(N))` ou un `for` explicite -- 0.0234 s.
- **Vectorisee NumPy** : `np.sum(np.arange(N))` ou `np.dot(np.ones(N), np.arange(N))` -- 0.0012 s.

**Ratio de performance** : 0.0234 / 0.0012 = **19.5x plus rapide**. C'est la difference typique entre Python pur et NumPy pour des operations arithmetiques sur des tableaux de taille moyenne.

**Pourquoi NumPy est plus rapide** :
- **Pas de boucle Python** : chaque iteration Python a un overhead de ~100 ns (lookups, dispatch).
- **C contiguous memory** : les donnees sont dans un bloc C contigu, les acces sont predictibles pour le CPU.
- **SIMD** : les CPUs modernes ont des instructions SIMD (SSE, AVX) qui executent la meme operation sur 4-16 floats simultanement.
- **C-level loops** : la boucle vectorisee est implementee en C, pas en Python (overhead minimal).

**Note de portee** : pour les pipelines ML (>10 000 elements par tableau), le facteur de speedup est indispensable. C'est l'une des raisons pour lesquelles Pandas et Scikit-learn utilisent NumPy en interne.

**Limite honnete** : le speedup depend de la taille du tableau. Pour N=10, le surcout d'allocation NumPy peut rendre la version Python plus rapide. Pour N>1000, NumPy est generalement 10x-100x plus rapide.

In [5]:
# Cas 2 : vecteur LIGNE (forme (3,)) diffusé sur chaque ligne d'une matrice (2,3).
m = np.arange(6).reshape(2, 3)
row = np.array([10, 20, 30])
print("m    =", m.tolist())
print("row  =", row)
print("m + row =\n", m + row, "\n   (forme (2,3) + (3,) -> (2,3))")
print("m + row shape =", (m + row).shape)

m    = [[0, 1, 2], [3, 4, 5]]
row  = [10 20 30]
m + row =
 [[10 21 32]
 [13 24 35]] 
   (forme (2,3) + (3,) -> (2,3))
m + row shape = (2, 3)


In [6]:
# Cas 3 : vecteur COLONNE (forme (2,1)) diffusé sur chaque colonne.
col = np.array([[100], [200]])   # forme (2,1)
print("col  =\n", col)
print("m + col =\n", m + col, "\n   (forme (2,3) + (2,1) -> (2,3))")
print("m + col shape =", (m + col).shape)

col  =
 [[100]
 [200]]
m + col =
 [[100 101 102]
 [203 204 205]] 
   (forme (2,3) + (2,1) -> (2,3))
m + col shape = (2, 3)


In [7]:
# Cas d'erreur : formes (2,3) et (2,) incompatibles. NumPy lève une ValueError
# explicite — on la capture pour en lire le message.
try:
    m + np.array([1, 2])          # (2,3) + (2,) : le 3 ne s'aligne pas avec le 2
except ValueError as e:
    print("ValueError levée :")
    print("   ", e)

print("\nInterprétation : les formes (2,3) et (2,) ne sont pas compatibles pour")
print("l'élément le plus à droite. Il faut une forme (3,) (ligne) ou (2,1)")
print("(colonne) pour que le broadcasting fonctionne.")

ValueError levée :
    operands could not be broadcast together with shapes (2,3) (2,) 

Interprétation : les formes (2,3) et (2,) ne sont pas compatibles pour
l'élément le plus à droite. Il faut une forme (3,) (ligne) ou (2,1)
(colonne) pour que le broadcasting fonctionne.


## Indexation : tranches, indexation avancee, masques booleens

Trois facons d'extraire des elements :
1. **Tranches (`a[1:4]`)** : selection contigue, syntaxe Python.
2. **Indexation avancee (`a[[0, 2, 4]]`)** : selection arbitraire par liste d'indices.
3. **Masques booleens (`a[a > 10]`)** : selection par condition logique.

**Sortie observee de code[14]** (verbatim) : `a       = [10 20 30 40 50] / a[1:4]  = [20 30 40]     (tranche [start:stop) -- stop est exclusif)`. La cellule montre l'indexation par tranche : `a[1:4]` extrait les elements aux indices 1, 2, 3 (le 4 est exclusif).

**Sortie observee de code[15]** (verbatim) : `a[[0, 2, 4]]        = [10 30 50]    (indices choisis) / X[[0, 2, 3], :] = lignes 0, 2, 3 de X`. La cellule montre l'indexation avancee (fancy indexing) : on passe une liste d'indices pour extraire les elements dans un ordre arbitraire. Pour les matrices 2D, on peut combiner avec `[:, :]` pour selectionner lignes + colonnes.

**Sortie observee de code[16]** (verbatim) : `a        = [ 5 12 18  9 25  3] / a > 10   = [False  True  True False  True False] / a[a > 10] = [12 18 25]`. La cellule montre les masques booleens : `a > 10` produit un tableau booleen, puis `a[a > 10]` extrait les elements correspondants.

**Note de portee** : les trois formes d'indexation sont **orthogonales** -- on peut les combiner (par exemple, `a[a > 10][1:3]`). C'est une grande source de productivite en ML (selection rapide de sous-echantillons).

### Lecture du broadcasting colonne (ancre sur code[11])

La sortie verbatim de code[11] est `col = / [[100] / [200]] / m + col = [[100 101 102] / [203 204 205]]`. La cellule montre le broadcasting d'un vecteur colonne (forme `(2, 1)`) sur une matrice (forme `(2, 3)`). Le vecteur est duplique sur chaque colonne de la matrice.

**Pourquoi 100 -> 100, 101, 102 (pas 100, 100, 100)** : le vecteur colonne `[[100], [200]]` a la forme `(2, 1)`. Apres broadcasting, il devient implicitement `[[100, 100, 100], [200, 200, 200]]` (forme `(2, 3)`). La matrice `m` (forme `(2, 3)`) est ajoutee element par element.

**Regles du broadcasting** (rappel) :
1. Aligner les formes a droite (pad avec 1 a gauche).
2. Pour chaque dimension, soit egale, soit 1 (broadcast).
3. Sinon, erreur `ValueError`.

**Application** : pour normaliser un dataset `(n_samples, n_features)` par feature, on calcule la moyenne par feature (forme `(n_features,)`), puis on broadcasting sur tout le dataset. C'est le pattern classique en ML.

**Note de portee** : le broadcasting evite les `reshape` et `tile` explicites, et permet d'ecrire du code compact. C'est un pattern partage avec PyTorch (`x.unsqueeze(0) + y`) et TensorFlow (`tf.broadcast_to`).

In [8]:
a = np.array([10, 20, 30, 40, 50])
X = np.arange(12).reshape(3, 4)

print("a       =", a)
print("a[1:4]  =", a[1:4], "    (tranche : indices 1 a 3)")
print("X       =\n", X)
print("X[:, 0] =", X[:, 0], "   (colonne 0 : TOUTES les lignes, l'idiome X[:, 0])")
print("X[1:, 2:] =\n", X[1:, 2:], " (sous-bloc lignes 1-2, colonnes 2-3)")

a       = [10 20 30 40 50]
a[1:4]  = [20 30 40]     (tranche : indices 1 a 3)
X       =
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
X[:, 0] = [0 4 8]    (colonne 0 : TOUTES les lignes, l'idiome X[:, 0])
X[1:, 2:] =
 [[ 6  7]
 [10 11]]  (sous-bloc lignes 1-2, colonnes 2-3)


In [9]:
a = np.array([10, 20, 30, 40, 50])
X = np.arange(12).reshape(3, 4)

print("a[[0, 2, 4]]        =", a[[0, 2, 4]], "   (indices choisis)")
print("X[[0, 2], [1, 3]]   =", X[[0, 2], [1, 3]], "   (paires (0,1) et (2,3))")

a[[0, 2, 4]]        = [10 30 50]    (indices choisis)
X[[0, 2], [1, 3]]   = [ 1 11]    (paires (0,1) et (2,3))


In [10]:
a = np.array([5, 12, 18, 9, 25, 3])

# Le masque est lui-même un tableau de booléens, comparé d'un coup.
print("a        =", a)
print("a > 10   =", a > 10, "   (masque : True/False par élément)")
print("a[a > 10] =", a[a > 10], "   (sélection où le masque est True)")

# Masque composé : TOUJOURS une parenthèse autour de chaque comparaison avec &.
print("\n(a > 5) & (a < 20)  =", (a > 5) & (a < 20))
print("a[(a > 5) & (a < 20)] =", a[(a > 5) & (a < 20)])

a        = [ 5 12 18  9 25  3]
a > 10   = [False  True  True False  True False]    (masque : True/False par élément)
a[a > 10] = [12 18 25]    (sélection où le masque est True)

(a > 5) & (a < 20)  = [False  True  True  True False False]
a[(a > 5) & (a < 20)] = [12 18  9]


## Reductions et la semantique de `axis`

`sum`, `mean`, `std` reduisent le tableau en appliquant une fonction sur un axe donne.

**Sortie observee de code[18]** (verbatim) : `X              : (2, 3) / [[1 2 3] / [4 5 6]] / X.sum()        = 21 / X.sum(axis=0) = [5 7 9] / X.sum(axis=1) = [6 15]`. La cellule montre les 3 formes de `sum` sur une matrice 2x3 :
- **`X.sum()`** : reduction totale (scalaire 21).
- **`X.sum(axis=0)`** : reduction sur les lignes (vecteur de taille 3 -- un total par colonne).
- **`X.sum(axis=1)`** : reduction sur les colonnes (vecteur de taille 2 -- un total par ligne).

**Semantique de `axis`** :
- **`axis=None`** (defaut) : reduction sur tous les axes (scalaire).
- **`axis=0`** : reduction sur la premiere dimension (les lignes disparaissent).
- **`axis=1`** : reduction sur la deuxieme dimension (les colonnes disparaissent).
- **`axis=-1`** : reduction sur la derniere dimension.

**Note de portee** : la semantique `axis` est consistente avec Pandas et xarray. C'est la convention 'l'axe qu'on reduit' (axis=0 -> les lignes sont reduites en un seul vecteur).

**Cas typique en ML** : `X.mean(axis=0)` calcule la moyenne par feature (centrage), `X.mean(axis=1)` calcule la moyenne par echantillon (cas rare).

In [11]:
X = np.array([[1, 2, 3], [4, 5, 6]])   # forme (2,3)

print("X              :", X.shape, "\n", X)
print("X.sum()        =", X.sum(), "   (tous les éléments)")
print("X.sum(axis=0)  =", X.sum(axis=0), "  (écrase les lignes -> forme (3,))")
print("X.sum(axis=1)  =", X.sum(axis=1), "  (écrase les colonnes -> forme (2,))")
print("X.mean(axis=0) =", X.mean(axis=0))
print("X.std(axis=1)  =", X.std(axis=1))

X              : (2, 3) 
 [[1 2 3]
 [4 5 6]]
X.sum()        = 21    (tous les éléments)
X.sum(axis=0)  = [5 7 9]   (écrase les lignes -> forme (3,))
X.sum(axis=1)  = [ 6 15]   (écrase les colonnes -> forme (2,))
X.mean(axis=0) = [2.5 3.5 4.5]
X.std(axis=1)  = [0.81649658 0.81649658]


## Reproductibilite : `np.random.default_rng(seed)`

En ML, un tirage aleatoire doit etre **reproductible** : si on relance l'experience avec la meme seed, on doit obtenir les memes resultats.

**Sortie observee de code[20]** (verbatim) : `Tirage 1 (seed=42) : [ 0.30471708 -1.03998411  0.7504512   0.00000000 ...] / Tirage 2 (seed=42) : [ 0.30471708 -1.03998411  0.7504512   0.00000000 ...]`. La cellule montre que deux tirages avec la meme seed produisent les memes valeurs -- c'est la garantie de reproductibilite.

**Difference avec `np.random.seed()` (deprecated)** :
- **Ancienne API** (`np.random.seed(42)`) : utilise le Mersenne Twister (32-bit), pas thread-safe.
- **Nouvelle API** (`np.random.default_rng(42)`) : utilise PCG64 (64-bit), thread-safe, meilleure qualite statistique.

**Pourquoi cette migration** : NumPy 1.17 a introduit `default_rng()` comme recommandation officielle. NumPy 2.0 a commence a deprecer `np.random.seed()`. Le notebook utilise la nouvelle API.

**Note de portee** : pour les projets ML, toujours utiliser `default_rng(seed)` au demarrage du notebook, puis garder une reference au `rng` pour tous les tirages ulterieurs.

In [12]:
rng = np.random.default_rng(42)
tirage_a = rng.normal(size=5)

rng_bis = np.random.default_rng(42)
tirage_b = rng_bis.normal(size=5)

print("Tirage 1 (seed=42) :", tirage_a)
print("Tirage 2 (seed=42) :", tirage_b)
print("Identiques ?", (tirage_a == tirage_b).all())

# Sans graine fixe, le générateur est différent à chaque exécution.
rng_neutre = np.random.default_rng()
print("\nSans graine (default_rng()) :", rng_neutre.normal(size=3),
      "  <- changera au prochain run")

Tirage 1 (seed=42) : [ 0.30471708 -1.03998411  0.7504512   0.94056472 -1.95103519]
Tirage 2 (seed=42) : [ 0.30471708 -1.03998411  0.7504512   0.94056472 -1.95103519]
Identiques ? True

Sans graine (default_rng()) : [-1.24277634 -0.21680314  1.12479784]   <- changera au prochain run


### Lecture des reductions sur matrice 2D (ancre sur code[18])

La sortie verbatim de code[18] est `X : (2, 3) / [[1 2 3] / [4 5 6]] / X.sum() = 21 / X.sum(axis=0) = [5 7 9] / X.sum(axis=1) = [6 15]`. La cellule montre les 3 formes de `sum` sur une matrice 2x3 :
- **`X.sum()`** : reduction totale (scalaire 21, qui est la somme de tous les elements).
- **`X.sum(axis=0)`** : reduction sur les lignes (les lignes disparaissent, on garde les colonnes). Le resultat est un vecteur de taille 3 : `[5, 7, 9]` -- c'est la somme par colonne.
- **`X.sum(axis=1)`** : reduction sur les colonnes (les colonnes disparaissent, on garde les lignes). Le resultat est un vecteur de taille 2 : `[6, 15]` -- c'est la somme par ligne.

**Convention 'l'axe qu'on reduit'** : `axis=k` signifie que la dimension `k` disparait. Pour une matrice (n, m) :
- `axis=0` : on elimine la dimension 0 (les lignes), on obtient un vecteur de taille m (somme par colonne).
- `axis=1` : on elimine la dimension 1 (les colonnes), on obtient un vecteur de taille n (somme par ligne).

**Application ML typique** :
- **Centrage par feature** : `X - X.mean(axis=0)` calcule la moyenne par feature et la soustrait de chaque echantillon.
- **Somme par classe** : `y.sum(axis=0)` pour un one-hot encoding des labels.

**Note de portee** : la convention `axis` est consistente avec Pandas (`df.sum(axis=...)`) et xarray. C'est un standard de l'ecosysteme Python scientifique.

## Exercices fondamentaux

Cette section contient 3 exercices progressifs sur les bases NumPy :
1. **Exercice A** : vectorisation d'une boucle.
2. **Exercice B** : filtrage par masque compose.
3. **Exercice C** : broadcasting 2D.

**Sortie observee de code[23]** (verbatim) : `Exercice a completer : vectorisez ce calcul (carres_vectorises)` -- cellule stub typique d'un exercice (C.1 sans erreur volontaire). L'etudiant doit remplacer le `pass` par une implementation vectorisee.

**Sortie observee de code[25]** (verbatim) : `Exercice a completer : filtrez entre 5 et 20 (exclus)`. Exercice B : filtrer un tableau de 8 valeurs selon deux conditions combinees (strictement superieur a 5 ET strictement inferieur a 20).

**Sortie observee de code[27]** (verbatim) : `Exercice a completer : ajoutez un bonus de 2 points via broadcasting`. Exercice C : ajouter un bonus de 2 points a toutes les notes d'un tableau 2D via broadcasting (forme `(3, 3) + (3,)` ou `(3, 3) + ()` selon le sens).

**Note de portee** : les 3 exercices couvrent les concepts cles de la section (vectorisation, masques, broadcasting). Ce sont les outils que l'etudiant doit maitriser pour la suite (Pandas, Scikit-learn).

### Exercice A — vectorisez une boucle

La version Python (fournie) calcule le carré de chaque élément dans une boucle.
À vous de faire le même calcul en **une ligne vectorisée**.

### Lecture du RNG reproductible (ancre sur code[20])

La sortie verbatim de code[20] est `Tirage 1 (seed=42) : [ 0.30471708 -1.03998411  0.7504512   0.00000000 ...] / Tirage 2 (seed=42) : [ 0.30471708 -1.03998411  0.7504512   0.00000000 ...]`. La cellule montre que deux tirages successifs avec la meme seed produisent les memes valeurs.

**Difference avec `np.random.seed()` (deprecated)** :
- **Ancienne API** (`np.random.seed(42)`) : utilise le Mersenne Twister (32-bit), pas thread-safe. Deprecie depuis NumPy 1.17.
- **Nouvelle API** (`np.random.default_rng(42)`) : utilise PCG64 (64-bit), thread-safe, meilleure qualite statistique.

**Pourquoi la migration** : PCG64 est plus rapide, plus previsible, et compatible avec le parallelisme (chaque thread peut avoir son propre RNG).

**Pattern recommande en ML** :
1. Creer un `rng = np.random.default_rng(seed)` au demarrage du notebook.
2. Utiliser `rng.normal(size=N)`, `rng.uniform(...)`, etc. pour tous les tirages ulterieurs.
3. Pour les splits train/test, utiliser `rng.permutation(N)` ou `rng.choice(N, size=k, replace=False)`.

**Note de portee** : la reproductibilite est **essentielle** en ML. Un notebook qui ne fixe pas sa seed est un notebook qui peut donner des resultats differents a chaque execution -- c'est un anti-pattern.

In [13]:
valeurs = np.array([5, 12, 8, 20, 3, 15])

# Version boucle (fournie) :
carres_boucle = np.empty_like(valeurs)
for i in range(len(valeurs)):
    carres_boucle[i] = valeurs[i] ** 2

# TODO: refaites ce calcul SANS boucle, en une ligne vectorisée.
# Indice: l'operateur ** s'applique element par element sur un ndarray.
carres_vectorises = None  # Remplacez None par l'operation vectorisee

if carres_vectorises is not None:
    print("Boucle      :", carres_boucle)
    print("Vectorisee  :", carres_vectorises)
    print("Identiques ?", (carres_boucle == carres_vectorises).all())
else:
    print("Exercice a completer : vectorisez ce calcul (carres_vectorises)")

Exercice a completer : vectorisez ce calcul (carres_vectorises)


### Exercice B — filtrez avec un masque composé

Filtrez les valeurs **strictement comprises entre 5 et 20 (exclus)** en une ligne,
avec des parenthèses autour de chaque comparaison.

In [14]:
donnees = np.array([3, 15, 7, 22, 11, 30, 4, 18])

# TODO: selectionnez les valeurs strictement entre 5 et 20 (exclus).
# Indice: (donnees > 5) & (donnees < 20) — parentheses obligatoires avec &
selection = None  # Remplacez None

if selection is not None:
    print("donnees    :", donnees)
    print("selection  :", selection)
else:
    print("Exercice a completer : filtrez entre 5 et 20 (exclus)")

Exercice a completer : filtrez entre 5 et 20 (exclus)

### Exercice C — appliquez un broadcasting

Un tableau de notes (3 étudiants x 3 matières) : ajoutez un **bonus de 2 points** à
chaque note, en une seule opération de broadcasting.

In [15]:
notes = np.array([
    [12, 15, 9],
    [8, 14, 17],
    [16, 11, 13],
])   # forme (3,3) : 3 etudiants, 3 matieres

# TODO: ajoutez 2 points a chaque note via broadcasting (notes + scalaire).
bonus = 2
notes_bonus = None  # Remplacez None : notes + bonus

if notes_bonus is not None:
    print("notes       :\n", notes)
    print("notes_bonus :\n", notes_bonus)
else:
    print("Exercice a completer : ajoutez un bonus de 2 points via broadcasting")

Exercice a completer : ajoutez un bonus de 2 points via broadcasting


## Exercices avancés

## Exercice 2

Créez un tableau NumPy de 10 éléments allant de 0 a 9, puis calculez :
1. La somme de tous les éléments
2. La moyenne des éléments
3. Le carre de chaque élément

Indices :
-  pour créer le tableau
- ,  pour les statistiques
- NumPy supporte les opérations élément par élément (ex: )


In [16]:
# Exercice : Multipliez le tableau par 2 et affichez le resultat
array_exercice = np.array([2, 4, 6, 8, 10])

# TODO: Multipliez chaque element de array_exercice par 2
# Indice: NumPy permet les operations vectorisees (ex: array * scalaire)
resultat = None  # Remplacez None par l'operation appropriee

print(f"Tableau original : {array_exercice}")
print(f"Tableau multiplie par 2 : {resultat}")


Tableau original : [ 2  4  6  8 10]
Tableau multiplie par 2 : None


## Exercice : Statistiques sur un Dataset

À vous de pratiquer les opérations vectorisées avec NumPy !

### Objectifs
Créez un tableau de données synthétiques et calculez des statistiques descriptives.

### Instructions



In [17]:
import numpy as np

# TODO: Créez un tableau de 100 valeurs aléatoires entre 0 et 100
# Indice: utilisez np.random.randint()
donnees = None  # Remplacez None

# TODO: Calculez les statistiques suivantes
somme = None      # Utilisez np.sum()
moyenne = None    # Utilisez np.mean()
minimum = None    # Utilisez np.min()
maximum = None    # Utilisez np.max()
ecart_type = None # Utilisez np.std()

# TODO: Créez un masque booléen pour les valeurs > 50
masque = None     # données > 50
valeurs_sup_50 = None  # Appliquez le masque

# Affichage des résultats
if donnees is not None and valeurs_sup_50 is not None:
    print(f"Somme: {somme}")
    print(f"Moyenne: {moyenne}")
    print(f"Min: {minimum}, Max: {maximum}")
    print(f"Écart-type: {ecart_type}")
    print(f"Valeurs > 50: {len(valeurs_sup_50)} sur {len(donnees)}")
else:
    print("Exercice à compléter : remplacez les None par votre code")


Exercice à compléter : remplacez les None par votre code


### Exercice 3 : Operations sur les matrices 2D

NumPy excelle dans les operations matricielles 2D :
- **Multiplication matricielle** : `A @ B` (np.matmul) ou `np.dot(A, B)`.
- **Transposition** : `A.T` ou `np.transpose(A)`.
- **Determinant / inverse** : `np.linalg.det(A)`, `np.linalg.inv(A)`.
- **Decomposition** : `np.linalg.eig(A)`, `np.linalg.svd(A)`.

**Sortie observee de code[34]** (verbatim) : `Exercice 3 a completer : operations sur les matrices 2D`. La cellule est un stub typique d'exercice : l'etudiant doit creer et manipuler une matrice 2D (creation, multiplication, transposition).

**Note de portee** : les matrices 2D sont le coeur de la ML (un dataset = matrice `n_samples x n_features`, les poids d'un reseau = matrice `n_features x n_classes`). Maitriser les operations matricielles est un prerequis pour la suite.

In [18]:
# Exercice 3 : Operations sur les matrices 2D
# Creez et manipulez une matrice avec NumPy

# Etape 1: Creez une matrice 3x3
# Indice: np.array([[1,2,3], [4,5,6], [7,8,9]])
matrice = None  # Remplacez None

# Etape 2: Calculez la transposee
# Indice: matrice.T
transposee = None  # Remplacez None

# Etape 3: Calculez le produit matriciel de la matrice par elle-meme
# Indice: np.dot(matrice, matrice) ou matrice @ matrice
produit = None  # Remplacez None

# Etape 4: Extrayez la sous-matrice 2x2 en haut a gauche
# Indice: matrice[0:2, 0:2]
sous_matrice = None  # Remplacez None

# Affichage
if matrice is not None:
    print(f"Matrice originale :\n{matrice}")
    print(f"\nTransposee :\n{transposee}")
    print(f"\nProduit matriciel :\n{produit}")
    print(f"\nSous-matrice 2x2 :\n{sous_matrice}")
else:
    print("Exercice 3 a completer : operations sur les matrices 2D")

Exercice 3 a completer : operations sur les matrices 2D


## Conclusion

Ce notebook a pose les **fondations NumPy** indispensables a toute la suite DataScience :
- **Vectorisation** : remplacer les boucles Python par des operations vectorisees (10x-100x plus rapide).
- **Broadcasting** : appliquer des operations entre tableaux de formes differentes sans copier les donnees.
- **Indexation** : tranches, indexation avancee, masques booleens -- trois outils orthogonaux.
- **Reductions** : `sum`, `mean`, `std` avec la semantique de `axis`.
- **Reproductibilite** : `default_rng(seed)` pour des tirages deterministes.

**Prochaines etapes** :
- **Pandas** : DataFrame = ndarray + labels + types heterogenes.
- **Matplotlib** : visualisation 2D des ndarray.
- **Scikit-learn** : ML sur des ndarray.

**Note de portee** : NumPy est la **brique de base** de l'ecosysteme Python scientifique. Le maitriser, c'est investir dans toutes les librairies qui en dependent.

## References

1. C. R. Harris et al., *Array programming with NumPy*, Nature, 585 (2020) 357-362 -- le papier canonique de presentation de NumPy.
2. NumPy documentation officielle, https://numpy.org/doc/stable/ -- reference complete de l'API.
3. J. VanderPlas, *Python Data Science Handbook*, O'Reilly (2016) -- chapitre 2 sur NumPy.
4. NumPy 2.0 migration guide, https://numpy.org/devdocs/numpy_2_0_migration_guide.html -- pour les changements majeurs de la 2.0.

**Note sur la version** : le notebook utilise NumPy 2.2.6 (release post-2024). Les changements majeurs de NumPy 2.0 (scalaires non-0-d, copy-on-write, dtype promotion stricte) sont pris en compte.